[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/try/)

# Operating modes: GEMMA, ISA-88 and PackML

Three ways of describing how a machine or a process runs and stops:

* **GEMMA**, the running and stopping modes study guide (ADEPA, 1981),
  widely taught in France: 16 procedures seen from the operative part, in
  three families, A (stop and restart), F (operation) and D (failure);
* **ISA-88 / IEC 61512-1** (batch control): the states and commands of a
  procedural element (a phase, an operation);
* **ISA-TR88.00.02 (PackML)**: the machine states of packaging machines and
  lines.

**Run all the cells**, then choose a **situation** in the list: the three models go to the matching
state through their own commands (the path taken is listed under the
table). You can also operate each model with its command buttons.

GEMMA describes modes (production, checks, tests) and states together;
ISA-88 and PackML separate the mode (automatic, manual, maintenance) from
the state within it, so the correspondence is approximate. The `D1`
emergency stop is a procedure of the software model only: the emergency
stop itself is a hardwired safety function.

**References:** GEMMA (ADEPA, 1981), ISA-88 / IEC 61512-1 (procedural
states), ISA-TR88.00.02 (PackML machine states). These references
inspired the widgets; the library does not claim conformity with them
([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments

In [ ]:
import anywidget_instruments as ai

ai.theme_switch()  # light / system / dark theme (STYLE-008)

In [ ]:
# situation: (title, GEMMA, ISA-88, PackML, note)
SITUATIONS = {
    "ready": ("Stopped, ready to start", "A1", "Idle", "Idle", ""),
    "startup": ("Start-up, warm-up", "F2", "Running", "Starting", ""),
    "production": ("Normal production", "F1", "Running", "Execute", ""),
    "waiting": (
        "Waiting for upstream or downstream",
        "F1",
        "Running",
        "Suspended",
        "Only PackML has a state for a starved or blocked machine.",
    ),
    "closing": (
        "Closing run (emptying)",
        "F3",
        "Running",
        "Completing",
        "ISA-88: the end of the procedure.",
    ),
    "end_of_cycle": ("Stop at the end of the cycle", "A2", "Complete", "Complete", ""),
    "given_state": ("Stop in a given state", "A4", "Paused", "Held", ""),
    "controlled": ("Controlled stop", "A3", "Stopped", "Stopped", ""),
    "checks": (
        "Checks and tests",
        "F5",
        "Idle",
        "Stopped",
        "ISA-88 and PackML: a manual or maintenance mode, not a state.",
    ),
    "diagnosis": ("Failure diagnosis", "D2", "Held", "Held", ""),
    "degraded": (
        "Running despite a failure",
        "D3",
        "Running",
        "Execute",
        "Not modeled by ISA-88 nor PackML.",
    ),
    "estop": ("Emergency stop", "D1", "Aborted", "Aborted", ""),
    "recovery": (
        "Back to the initial state after a failure",
        "A6",
        "Idle",
        "Resetting",
        "",
    ),
}
GROUPS = {
    "Running": ["startup", "production", "waiting", "closing", "degraded"],
    "Stopping": ["ready", "end_of_cycle", "given_state", "controlled"],
    "Checks": ["checks"],
    "Failures": ["diagnosis", "estop", "recovery"],
}
STATUS = {
    "production": "running",
    "degraded": "warning",
    "estop": "alarm",
    "diagnosis": "fault",
}


gemma = ai.StateMachine("gemma", label="GEMMA", size=(760, 400))
isa88 = ai.StateMachine("isa88", label="ISA-88 procedural states", size=(520, 280))
packml = ai.StateMachine("packml", label="PackML machine states", size=(560, 280))


def goto(machine, state):
    """Drive `machine` to `state` through its own commands; return the path."""
    path = machine.path_to(state) or []
    for command in path:
        if command == "SC":
            machine.state_complete()  # an acting state completes
        else:
            machine.command(command)
    return path

In [ ]:
situations = ai.EquipmentTree(
    nodes=[
        {
            "id": group,
            "label": group,
            "children": [
                {"id": key, "label": SITUATIONS[key][0], "status": STATUS.get(key, "")}
                for key in keys
            ],
        }
        for group, keys in GROUPS.items()
    ],
    value="ready",
    expanded=list(GROUPS),
    show_level=False,
    label="Situation",
    size=(300, 420),
)

In [ ]:
from IPython.display import Markdown, display


def show(key):
    """Drive the three models to a situation and describe it."""
    if key not in SITUATIONS:
        return  # a group is selected: keep the models where they are
    title, g, i, p, _note = SITUATIONS[key]
    paths = [goto(gemma, g), goto(isa88, i), goto(packml, p)]
    rows = [
        f"| {'▶ ' if k == key else ''}{t} | {sg} | {si} | {sp} | {n} |"
        for k, (t, sg, si, sp, n) in SITUATIONS.items()
    ]
    applied = "; ".join(
        f"{name}: {' → '.join(path) if path else 'already there'}"
        for name, path in zip(["GEMMA", "ISA-88", "PackML"], paths, strict=True)
    )
    summary.update(
        Markdown(
            f"**{title}**: GEMMA `{gemma.value}` {gemma.state_title}, "
            f"ISA-88 `{isa88.value}`, PackML `{packml.value}`.\n\n"
            f"Commands applied ('SC': an acting state completes): {applied}\n\n"
            "| Situation | GEMMA | ISA-88 | PackML | Note |\n|---|---|---|---|---|\n"
            + "\n".join(rows)
        )
    )


situations.on_change(lambda change: show(change["new"]))
display(ai.Panel([situations, gemma], columns=2))
display(ai.Panel([isa88, packml], columns=2))
summary = display(Markdown(""), display_id=True)
show(situations.value)